## Simple Agents

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

In [17]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    api_key = API_KEY, 
    base_url = BASE_URL, 
    model = 'gpt-5.1',
    temperature = 2
)

In [18]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model = llm, 
    system_prompt = "You are a helful assistant, always reply the user short and direct."
)

In [19]:
result = agent.invoke({'messages': HumanMessage(content='Tell me a joke on cricket')})

In [20]:
for msg in result['messages']:
    print(f'{msg.type.upper()}: {msg.content}')

HUMAN: Tell me a joke on cricket
AI: Why did the cricket ball apply for a job?

Because it was tired of being hit around and wanted a more “straight” career!


Alternative Formats

In [ ]:
result = agent.invoke({'messages':'"Write a funny joke about Panda."'})
print(result["messages"][-1].content)    

In [24]:
result = agent.invoke({'messages':{'role':'user', 'content': 'tell me a funny street joke on humans'}})
print(result["messages"][-1].content)    

Humans are the only creatures who pay to live on Earth, destroy it, then complain that the Wi-Fi is slow.


In [26]:
result = agent.invoke({'messages':('human', 'funny joke on laptop')})
print(result["messages"][-1].content)    

Why did the laptop go to therapy?

Because it had too many tabs open and couldn’t process its feelings.


### Streaming the values

`stream_mode = 'values'` will stream data after every step in the agent loop.

In [38]:
for step in agent.stream(
    {'messages': "Tell me a Dad joke"},
    stream_mode = 'values'
):
    step['messages'][-1].pretty_print()
    

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Why did the scarecrow win an award?  

Because he was outstanding in his field.


`stream_mode = 'messages'` - gets the data token by token - Lowest latency possible
(Best for interactive Chatbots)

In [40]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write short blog in 300 words on AgentAI in healthcare."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

AgentAI refers to autonomous, intelligent software agents that can perform complex tasks with minimal human intervention. In healthcare, these agents are transforming how care is delivered, decisions are made, and systems are managed.

One key application is clinical decision support. AgentAI can continuously scan patient data—vital signs, lab results, imaging, and history—to identify risks early, such as sepsis or heart failure. Unlike static rule-based systems, these agents learn and adapt, refining their recommendations as more data becomes available. This helps clinicians make faster, more accurate decisions while reducing cognitive overload.

Another powerful use is in workflow automation. AgentAI can coordinate appointments, triage patients based on symptoms, and route cases to the right specialists. In hospitals, agents can manage bed allocation, operating room schedules, and resource use in real time, improving efficiency and reducing wait times.

Patient engagement is also evo

`stream_mode="updates"` - For getting the update in each step

In [44]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Write short blog in 300 words on AgentAI in healthcare."}]},
    stream_mode="updates",
):
    print(f"{step['model']['messages']}", end="")

[AIMessage(content='AgentAI—autonomous software “agents” powered by advanced AI—are rapidly transforming healthcare. Unlike traditional systems that only respond to direct commands, AgentAI can perceive, reason, and act continuously, making proactive decisions within defined boundaries.\n\nIn clinical settings, AgentAI can automate routine administrative tasks such as scheduling, prior authorizations, and documentation. By integrating with electronic health records, agents can pull patient histories, summarize key information, and prepare drafts of clinical notes, freeing clinicians to focus more on patient care and less on paperwork.\n\nIn diagnostics, AgentAI can assist radiologists and pathologists by pre-screening images, highlighting suspicious areas, and prioritizing urgent cases. These agents don’t replace specialists; they act as tireless assistants, reducing error rates and speeding time to diagnosis, especially in high-volume environments.\n\nFor chronic disease management, A